# Delta Lake — Controle de produtos de uma loja

Este notebook demonstra o uso do Delta Lake com PySpark para armazenar
e modificar dados de uma loja.

O cenário possui duas tabelas: categorias e produtos. Uma categoria
pode ter vários produtos, e cada produto pertence a uma categoria.

Os dados são fictícios e foram criados pelo grupo. Serão demonstradas
as operações INSERT, UPDATE e DELETE, com consultas para verificar
os resultados.

## 1. Preparação do ambiente

Nesta etapa, iniciamos uma sessão Spark com suporte ao Delta Lake.
O processamento será executado localmente.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Loja - Delta Lake")
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Versão do Spark:", spark.version)

26/10/05 21:00:32 WARN Utils: Your hostname, YasminNitroV15 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/05 21:00:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/bfontanayasmin/projetos/trabalhoBancodeDados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/bfontanayasmin/.ivy2/cache
The jars for the packages stored in: /home/bfontanayasmin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8e6d63a4-28b6-4492-9fea-3b67f5bf0f2d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 188ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

Versão do Spark: 3.5.3


### 1.1 Limpeza do exemplo

Esta etapa prepara o ambiente para executar o notebook novamente desde
o início, removendo as tabelas e os dados de uma execução anterior.

O código realiza as seguintes ações:

1. Usa Path para localizar a pasta do projeto a partir da pasta notebooks.
2. Verifica a existência do arquivo pyproject.toml para confirmar o local
   de execução.
3. Limpa os dados mantidos em cache pelo Spark.
4. Executa DROP DATABASE IF EXISTS loja_delta CASCADE para remover
   o banco loja_delta e as tabelas cadastradas nele.
5. Usa shutil.rmtree para excluir a pasta data/delta, incluindo os
   arquivos dos dados e o histórico de alterações.

IF EXISTS evita um erro caso o banco ainda não exista. A pasta de dados
também só é excluída quando existe.

Depois da limpeza, as próximas células criam as tabelas vazias e
demonstram novamente as operações INSERT, UPDATE e DELETE.

In [2]:
from pathlib import Path
import shutil

raiz_projeto = Path("..").resolve()

if not (raiz_projeto / "pyproject.toml").is_file():
    raise RuntimeError("Execute este notebook dentro da pasta notebooks.")

spark.catalog.clearCache()

# Exclui o banco e todas as tabelas cadastradas nele
spark.sql("DROP DATABASE IF EXISTS loja_delta CASCADE")

# Exclui os arquivos de dados e o histórico do exemplo Delta
pasta_delta = raiz_projeto / "data" / "delta"

if pasta_delta.exists():
    shutil.rmtree(pasta_delta)

print("Limpeza concluída. O exemplo Delta pode ser iniciado novamente.")

Limpeza concluída. O exemplo Delta pode ser iniciado novamente.


## 2. Estrutura das tabelas

A tabela categorias armazena o identificador e o nome de cada categoria.

| Campo | Tipo | Descrição |
|---|---|---|
| id_categoria | INT | Identificador da categoria |
| nome_categoria | STRING | Nome da categoria |

A tabela produtos armazena os produtos da loja.

| Campo | Tipo | Descrição |
|---|---|---|
| id_produto | INT | Identificador do produto |
| nome_produto | STRING | Nome do produto |
| preco | DECIMAL(10,2) | Preço com duas casas decimais |
| estoque | INT | Quantidade disponível |
| id_categoria | INT | Categoria à qual o produto pertence |

O relacionamento entre categorias e produtos é de um para muitos.
O campo id_categoria permite relacionar as duas tabelas. Neste exemplo,
essa associação é mantida pelos dados, sem uma restrição de chave estrangeira.

### Modelo entidade-relacionamento

![Modelo ER da loja](../docs/assets/modelo_er_loja.png)

O modelo representa uma relação de um para muitos (1:N):
uma categoria pode possuir vários produtos, e cada produto
pertence a uma categoria.

## 3. Criação das tabelas

Os comandos DDL abaixo definem as tabelas no formato Delta.

Usamos CREATE OR REPLACE TABLE para começar com tabelas vazias a cada
execução completa do notebook. Essa etapa reinicia os dados fictícios
do exemplo.

In [3]:
from pathlib import Path

pasta_delta = (Path("..") / "data" / "delta").resolve()
pasta_delta.mkdir(parents=True, exist_ok=True)

spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS loja_delta
    LOCATION '{pasta_delta.as_posix()}'
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE loja_delta.categorias (
        id_categoria INT,
        nome_categoria STRING
    )
    USING DELTA
    LOCATION '{(pasta_delta / "categorias").as_posix()}'
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE loja_delta.produtos (
        id_produto INT,
        nome_produto STRING,
        preco DECIMAL(10,2),
        estoque INT,
        id_categoria INT
    )
    USING DELTA
    LOCATION '{(pasta_delta / "produtos").as_posix()}'
""")

spark.sql("SHOW TABLES IN loja_delta").show()

+----------+----------+-----------+
| namespace| tableName|isTemporary|
+----------+----------+-----------+
|loja_delta|categorias|      false|
|loja_delta|  produtos|      false|
+----------+----------+-----------+



## 4. Inserção de dados — INSERT

O comando INSERT INTO adiciona registros a uma tabela.

Primeiro, cadastramos as categorias Informática e Papelaria.
Depois, cadastramos dez produtos, indicando a categoria de cada um.

As consultas antes e depois da inserção permitem verificar o resultado.

In [4]:
print("Antes da inserção:")
spark.sql("SELECT * FROM loja_delta.categorias").show()
spark.sql("SELECT * FROM loja_delta.produtos").show()

spark.sql("""
    INSERT INTO loja_delta.categorias
        (id_categoria, nome_categoria)
    VALUES
        (1, 'Informática'),
        (2, 'Papelaria')
""")

spark.sql("""
    INSERT INTO loja_delta.produtos
        (id_produto, nome_produto, preco, estoque, id_categoria)
    VALUES
        (1, 'Mouse', 49.90, 20, 1),
        (2, 'Teclado', 120.00, 10, 1),
        (3, 'Caderno', 19.90, 30, 2),
        (4, 'Caneta', 3.50, 100, 2),
        (5, 'Headset', 199.90, 12, 1),
        (6, 'Webcam', 149.90, 8, 1),
        (7, 'Pendrive', 39.90, 25, 1),
        (8, 'Lapiseira', 9.90, 40, 2),
        (9, 'Borracha', 2.50, 60, 2),
        (10, 'Marcador', 6.90, 35, 2)
""")

print("Depois da inserção:")
spark.sql("""
    SELECT * FROM loja_delta.categorias
    ORDER BY id_categoria
""").show()

spark.sql("""
    SELECT * FROM loja_delta.produtos
    ORDER BY id_produto
""").show()

Antes da inserção:


26/10/05 21:00:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+------------+--------------+
|id_categoria|nome_categoria|
+------------+--------------+
+------------+--------------+

+----------+------------+-----+-------+------------+
|id_produto|nome_produto|preco|estoque|id_categoria|
+----------+------------+-----+-------+------------+
+----------+------------+-----+-------+------------+



26/10/05 21:00:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/05 21:00:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/05 21:00:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/05 21:00:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/05 21:00:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Depois da inserção:
+------------+--------------+
|id_categoria|nome_categoria|
+------------+--------------+
|           1|   Informática|
|           2|     Papelaria|
+------------+--------------+

+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 49.90|     20|           1|
|         2|     Teclado|120.00|     10|           1|
|         3|     Caderno| 19.90|     30|           2|
|         4|      Caneta|  3.50|    100|           2|
|         5|     Headset|199.90|     12|           1|
|         6|      Webcam|149.90|      8|           1|
|         7|    Pendrive| 39.90|     25|           1|
|         8|   Lapiseira|  9.90|     40|           2|
|         9|    Borracha|  2.50|     60|           2|
|        10|    Marcador|  6.90|     35|           2|
+----------+------------+------+-------+------------+



## 5. Atualização de dados — UPDATE

O comando UPDATE modifica registros existentes.

Neste exemplo, alteramos o preço do Mouse de R$ 49,90 para R$ 59,90
e o estoque de 20 para 15 unidades.

A condição WHERE id_produto = 1 identifica o produto que será alterado.
Os demais produtos mantêm seus valores.

In [5]:
print("Antes da atualização:")
spark.sql("""
    SELECT * FROM loja_delta.produtos
    ORDER BY id_produto
""").show()

spark.sql("""
    UPDATE loja_delta.produtos
    SET preco = 59.90,
        estoque = 15
    WHERE id_produto = 1
""")

print("Depois da atualização:")
spark.sql("""
    SELECT * FROM loja_delta.produtos
    ORDER BY id_produto
""").show()

Antes da atualização:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 49.90|     20|           1|
|         2|     Teclado|120.00|     10|           1|
|         3|     Caderno| 19.90|     30|           2|
|         4|      Caneta|  3.50|    100|           2|
|         5|     Headset|199.90|     12|           1|
|         6|      Webcam|149.90|      8|           1|
|         7|    Pendrive| 39.90|     25|           1|
|         8|   Lapiseira|  9.90|     40|           2|
|         9|    Borracha|  2.50|     60|           2|
|        10|    Marcador|  6.90|     35|           2|
+----------+------------+------+-------+------------+

Depois da atualização:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 59.90|    

## 6. Exclusão de dados — DELETE

O comando DELETE FROM remove registros de uma tabela.

Neste exemplo, excluímos a Caneta, identificada por id_produto = 4.
A condição WHERE limita a exclusão a esse produto.

As consultas antes e depois mostram que os outros nove produtos
permanecem na tabela.

In [6]:
print("Antes da exclusão:")
spark.sql("""
    SELECT * FROM loja_delta.produtos
    ORDER BY id_produto
""").show()

spark.sql("""
    DELETE FROM loja_delta.produtos
    WHERE id_produto = 4
""")

print("Depois da exclusão:")
spark.sql("""
    SELECT * FROM loja_delta.produtos
    ORDER BY id_produto
""").show()

Antes da exclusão:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 59.90|     15|           1|
|         2|     Teclado|120.00|     10|           1|
|         3|     Caderno| 19.90|     30|           2|
|         4|      Caneta|  3.50|    100|           2|
|         5|     Headset|199.90|     12|           1|
|         6|      Webcam|149.90|      8|           1|
|         7|    Pendrive| 39.90|     25|           1|
|         8|   Lapiseira|  9.90|     40|           2|
|         9|    Borracha|  2.50|     60|           2|
|        10|    Marcador|  6.90|     35|           2|
+----------+------------+------+-------+------------+

Depois da exclusão:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 59.90|     15|  

26/10/05 21:02:39 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-e0f8ee1d-0894-4d81-8bb5-d4631536df5f/userFiles-a2be8226-d105-48f4-afc7-a840c2158839. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-e0f8ee1d-0894-4d81-8bb5-d4631536df5f/userFiles-a2be8226-d105-48f4-afc7-a840c2158839
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.SparkEnv.stop(SparkEnv.scala:108)
	at org.apache.spark.SparkContext.$anonfun$stop$25(SparkContext.scala:2305)
	at org.apac